# 04 — Akobo, South Sudan: Satellite Imagery HUD

This browser-native example opens a live **Esri World Imagery** satellite basemap centered on **Akobo, Jonglei State, South Sudan** (approximately **7.79293° N, 33.00294° E**).

The presentation is inspired by the satellite/globe demos in [Bilawal Sidhu's *God's Eye View*](https://github.com/bilawalsidhu/gods-eye-view), while the notebook implementation below is original and deliberately lightweight for JupyterLite.

**What the view adds**

- satellite imagery with pan/zoom controls;
- a HUD-style coordinate/zoom readout;
- a center reticle and 5/15/30 km range rings;
- NORMAL, MONO, NVG-like, and THERMAL-like browser display filters;
- explicit imagery attribution.

> The MONO/NVG-like/THERMAL-like modes are **display effects only**. They do not turn RGB basemap imagery into real multispectral, thermal, night-vision, or intelligence sensor data.

**Imagery attribution:** Powered by Esri — Source: Esri, Maxar, Earthstar Geographics, and the GIS User Community. Provider terms apply. This notebook fetches imagery at runtime and does not bundle or redistribute tiles.


In [ ]:
from html import escape
from IPython.display import HTML, display

AKOBO_LAT = 7.79293
AKOBO_LON = 33.00294
AKOBO_ZOOM = 13

print(f"Akobo view center: {AKOBO_LAT:.5f}° N, {AKOBO_LON:.5f}° E")
print("Imagery: Esri World Imagery (loaded live in the browser)")


In [ ]:
def akobo_satellite_hud(lat=AKOBO_LAT, lon=AKOBO_LON, zoom=AKOBO_ZOOM, height=720):
    '''Return an iframe containing a Leaflet + Esri World Imagery HUD view.'''
    page = f'''<!doctype html>
<html>
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"
  integrity="sha256-p4NxAoJBhIINfQ3ynQWZ1QC6+Q3P9G1y5B+MZ4O2X4A=" crossorigin="">
<style>
  html, body, #map {{ height: 100%; margin: 0; background: #020706; overflow: hidden; }}
  body {{ font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; }}
  #map {{ position: absolute; inset: 0; }}
  .leaflet-tile-pane {{ transition: filter .25s ease; }}
  .leaflet-control-attribution {{ background: rgba(0,0,0,.72)!important; color:#d7fff3!important; }}
  .leaflet-control-attribution a {{ color:#83ffe0!important; }}
  .leaflet-control-zoom a {{ background:#07120f!important; color:#77ffd9!important; border-color:#265b4e!important; }}
  .hud {{ position:absolute; inset:0; z-index:800; pointer-events:none; color:#7dffe0; text-shadow:0 0 7px #00d9a6; }}
  .panel {{ position:absolute; background:rgba(0,12,10,.70); border:1px solid rgba(95,255,213,.55); box-shadow:0 0 18px rgba(0,255,200,.08) inset; padding:10px 12px; letter-spacing:.08em; font-size:12px; line-height:1.55; }}
  .tl {{ top:14px; left:58px; min-width:265px; }}
  .tr {{ top:14px; right:14px; text-align:right; min-width:215px; }}
  .bl {{ left:14px; bottom:30px; }}
  .title {{ color:#d7fff6; font-weight:700; font-size:15px; letter-spacing:.14em; }}
  .muted {{ color:#76bfae; }}
  .reticle {{ position:absolute; left:50%; top:50%; width:58px; height:58px; transform:translate(-50%,-50%); border:1px solid rgba(109,255,221,.72); border-radius:50%; }}
  .reticle:before, .reticle:after {{ content:''; position:absolute; background:#82ffe3; box-shadow:0 0 8px #00d7a5; }}
  .reticle:before {{ width:92px; height:1px; left:-18px; top:28px; }}
  .reticle:after {{ width:1px; height:92px; top:-18px; left:28px; }}
  .corner {{ position:absolute; width:42px; height:42px; border-color:#69ffda; opacity:.75; }}
  .c1 {{ left:18px; top:18px; border-left:2px solid; border-top:2px solid; }}
  .c2 {{ right:18px; top:18px; border-right:2px solid; border-top:2px solid; }}
  .c3 {{ left:18px; bottom:18px; border-left:2px solid; border-bottom:2px solid; }}
  .c4 {{ right:18px; bottom:18px; border-right:2px solid; border-bottom:2px solid; }}
  .controls {{ position:absolute; right:14px; bottom:34px; z-index:900; display:flex; gap:6px; flex-wrap:wrap; justify-content:flex-end; max-width:420px; }}
  .controls button {{ pointer-events:auto; cursor:pointer; font:600 11px ui-monospace,monospace; letter-spacing:.08em; color:#aaffea; background:rgba(2,18,15,.9); border:1px solid #388a74; padding:8px 10px; border-radius:2px; }}
  .controls button:hover {{ background:#0c3128; }}
  .leaflet-tooltip.hud-label {{ background:rgba(0,10,8,.82); border:1px solid #59dabd; color:#d7fff4; box-shadow:none; font:600 11px ui-monospace,monospace; }}
</style>
</head>
<body>
<div id="map"></div>
<div class="hud">
  <div class="corner c1"></div><div class="corner c2"></div><div class="corner c3"></div><div class="corner c4"></div>
  <div class="reticle"></div>
  <div class="panel tl">
    <div class="title">AKOBO / SOUTH SUDAN</div>
    <div>AOI CENTER <span id="centerReadout"></span></div>
    <div>ZOOM <span id="zoomReadout"></span> &nbsp; | &nbsp; RANGE RINGS 5 / 15 / 30 KM</div>
    <div class="muted">JUPYTERLITE SATELLITE VIEW</div>
  </div>
  <div class="panel tr">
    <div>BASEMAP: ESRI WORLD IMAGERY</div>
    <div>MODE: <span id="modeReadout">NORMAL</span></div>
    <div class="muted">RGB VISUALIZATION / NOT LIVE SENSOR TASKING</div>
  </div>
  <div class="panel bl">CENTER RETICLE • PAN / ZOOM ENABLED</div>
</div>
<div class="controls">
  <button onclick="setMode('NORMAL')">NORMAL</button>
  <button onclick="setMode('MONO')">MONO</button>
  <button onclick="setMode('NVG-LIKE')">NVG-LIKE</button>
  <button onclick="setMode('THERMAL-LIKE')">THERMAL-LIKE</button>
  <button onclick="resetView()">RESET AOI</button>
</div>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"
  integrity="sha256-20nQCchB9co0qIjJZRGuk2/Z9VM+kNiyxNV1lvTlZBo=" crossorigin=""></script>
<script>
const AOI = [{lat}, {lon}];
const INITIAL_ZOOM = {zoom};
const map = L.map('map', {{ zoomControl:true, attributionControl:true, preferCanvas:true }}).setView(AOI, INITIAL_ZOOM);

const imagery = L.tileLayer(
  'https://services.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{{z}}/{{y}}/{{x}}',
  {{
    maxZoom: 19,
    attribution: '<a href="https://www.esri.com/" target="_blank" rel="noopener">Powered by Esri</a> — Source: Esri, Maxar, Earthstar Geographics, and the GIS User Community'
  }}
).addTo(map);

const ringStyle = {{ color:'#67ffe0', weight:1.2, opacity:.85, fill:false, dashArray:'6 8', interactive:false }};
[5000, 15000, 30000].forEach((r) => L.circle(AOI, {{...ringStyle, radius:r}}).addTo(map));

const target = L.circleMarker(AOI, {{radius:5, color:'#cafff3', weight:2, fillColor:'#00d9a6', fillOpacity:.85}}).addTo(map);
target.bindTooltip('AKOBO AOI • 7.79293 N / 33.00294 E', {{permanent:false, direction:'top', className:'hud-label'}});

L.control.scale({{imperial:false, maxWidth:150}}).addTo(map);

function updateReadout() {{
  const c = map.getCenter();
  document.getElementById('centerReadout').textContent = `${{c.lat.toFixed(5)}} N / ${{c.lng.toFixed(5)}} E`;
  document.getElementById('zoomReadout').textContent = map.getZoom();
}}
map.on('move zoom', updateReadout);
updateReadout();

const filters = {{
  'NORMAL': 'none',
  'MONO': 'grayscale(1) contrast(1.35) brightness(.9)',
  'NVG-LIKE': 'grayscale(1) sepia(.55) hue-rotate(70deg) saturate(5) contrast(1.45) brightness(.78)',
  'THERMAL-LIKE': 'grayscale(1) invert(.1) sepia(.85) saturate(6) hue-rotate(305deg) contrast(1.7) brightness(.9)'
}};
window.setMode = function(name) {{
  document.querySelector('.leaflet-tile-pane').style.filter = filters[name];
  document.getElementById('modeReadout').textContent = name;
}};
window.resetView = function() {{ map.flyTo(AOI, INITIAL_ZOOM, {{duration:.8}}); }};
</script>
</body>
</html>'''

    srcdoc = escape(page, quote=True)
    return HTML(
        f'<iframe srcdoc="{srcdoc}" '
        f'style="width:100%;height:{int(height)}px;border:1px solid #1d5549;background:#020706" '
        f'loading="eager" allowfullscreen></iframe>'
    )


display(akobo_satellite_hud())


## Notes and provenance

- **Location center:** approximately 7.79293° N, 33.00294° E, corresponding to the mapped Akobo town point in OpenStreetMap/GeoNames-derived references.
- **Imagery:** Esri World Imagery is requested directly by the browser at runtime. No imagery tiles are stored in this repository.
- **Upstream relationship:** *God's Eye View* uses Esri World Imagery as its keyless satellite basemap and provides the same Esri/provider credit in its imagery implementation. This notebook uses Leaflet rather than copying the upstream Cesium code.
- **Limitations:** imagery dates and resolution vary by location and zoom level. The visual filter buttons are CSS effects only and should not be interpreted as actual thermal, infrared, or night-vision observations.

Try panning east toward the South Sudan–Ethiopia border, zooming into the town, or switching the visual mode buttons while keeping the Esri attribution visible.
